# Etapa 3 - Baseline e estrategia algoritmica (multi-armed bandit)

**Datathon 7MLET - Grupo 87**

Compara uma **politica simples (baseline)** com abordagens de **multi-armed bandit**
(Thompson Sampling e UCB1), medindo **recompensa, regret, exploracao e conversao**, e
analisa **cold-start** e **recompensas atrasadas**.

Algoritmos:
- **Baseline fixo (controle):** regra deterministica, sempre o braco de controle.
- **Epsilon-greedy (eps=0.1):** baseline adaptativo simples.
- **UCB1 (familia "Nilos-UCB"):** `mu_a + sqrt(2 ln t / n_a)` — explotacao + bonus de incerteza.
- **Thompson Sampling:** bayesiano Beta-Bernoulli (prior Beta(1,1)).

Tratamentos:
- **Cold-start:** TS via prior Beta(1,1); UCB jogando cada braco uma vez; eps-greedy pelo ramo aleatorio.
- **Recompensas atrasadas:** observacao chega `d ~ Poisson(media)` rodadas apos a decisao; a politica so atualiza ao receber.

> A camada de codigo vive em `src/bandits/`. O relatorio completo e gerado por
> `poetry run python -m src.bandits.experiment` (`reports/bandit-comparison.md`).


In [1]:
import os
import sys
from pathlib import Path

_p = Path.cwd()
while _p != _p.parent and not (_p / "pyproject.toml").exists():
    _p = _p.parent
PROJECT_ROOT = _p
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.bandits.environment import build_environment_from_catalog
from src.bandits.experiment import policy_factories
from src.bandits.simulation import run_policy

plt.rcParams["figure.dpi"] = 110
HORIZON, SEEDS, MEAN_DELAY = 20000, 20, 500
print("Projeto:", PROJECT_ROOT)


Projeto: D:\5. Academia\4. Machine Learning Engineering - FIAP\projeto_5\datathon-7mlet-grupo-87


## 1. Ambiente de simulacao

In [2]:
env = build_environment_from_catalog()
env_df = pd.DataFrame({
    "arm_id": env.arm_ids,
    "arm_name": env.arm_names,
    "true_p": env.true_p,
}).assign(otimo=lambda d: d.index == env.best_arm)
display(env_df)
print(f"Braco otimo: {env.arm_names[env.best_arm]} (p={env.best_p:.3f})")


,arm_id,arm_name,true_p,otimo
0,arm_consultative_call,Contato consultivo,0.145,False
1,arm_control,Mensagem neutra,0.040,False
2,arm_digital_bundle,Pacote digital,0.110,False
3,arm_rate_boost,Taxa bonificada,0.180,True
4,arm_retention_plus,Oferta de relacionamento,0.075,False


Braco otimo: Taxa bonificada (p=0.180)


## 2. Comparacao principal (feedback imediato)

In [3]:
results = [run_policy(f, env, HORIZON, SEEDS, mean_delay=0.0) for f in policy_factories(env)]

metrics = pd.DataFrame([
    {
        "politica": r.name,
        "recompensa": round(r.final_reward_mean),
        "conversao_%": round(r.conversion_rate * 100, 2),
        "regret_final": round(r.final_regret_mean, 1),
        "exploracao_%": round(r.exploration_rate * 100, 1),
        "%_braco_otimo": round(r.optimal_arm_rate * 100, 1),
    }
    for r in results
])
display(metrics)


,politica,recompensa,conversao_%,regret_final,exploracao_%,%_braco_otimo
0,Baseline fixo (controle),803,4.01,2800.0,0.0,0.0
1,Epsilon-greedy (eps=0.1),3381,16.91,209.7,8.0,82.2
2,UCB1 (Nilos-UCB),3173,15.86,426.1,33.9,65.9
3,Thompson Sampling,3533,17.67,66.5,5.5,94.1


### Regret acumulado

In [4]:
plt.figure(figsize=(9, 5))
for r in results:
    plt.plot(r.mean_cum_regret, label=r.name)
    plt.fill_between(range(HORIZON), r.mean_cum_regret - r.std_cum_regret,
                     r.mean_cum_regret + r.std_cum_regret, alpha=0.12)
plt.xlabel("rodada"); plt.ylabel("regret acumulado")
plt.title("Regret acumulado (media +/- desvio)"); plt.legend(); plt.show()


C:\Users\Narcelio Filho\AppData\Local\Temp\ipykernel_27908\1975126050.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Regret acumulado (media +/- desvio)"); plt.legend(); plt.show()


### Recompensa acumulada vs oraculo

In [5]:
plt.figure(figsize=(9, 5))
for r in results:
    plt.plot(r.mean_cum_reward, label=r.name)
plt.plot(range(HORIZON), env.best_p * np.arange(1, HORIZON + 1), "k--", lw=1, label="oraculo")
plt.xlabel("rodada"); plt.ylabel("recompensa acumulada")
plt.title("Recompensa acumulada"); plt.legend(); plt.show()


C:\Users\Narcelio Filho\AppData\Local\Temp\ipykernel_27908\805990436.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Recompensa acumulada"); plt.legend(); plt.show()


### Distribuicao de selecao por braco

In [6]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(env.n_arms); width = 0.8 / len(results)
for i, r in enumerate(results):
    ax.bar(x + i * width, r.arm_distribution * 100, width, label=r.name)
ax.set_xticks(x + 0.4 - width / 2)
ax.set_xticklabels([f"{n}\n(p={p:.3f})" for n, p in zip(env.arm_names, env.true_p)], fontsize=8, rotation=15)
ax.axvline(env.best_arm + 0.4 - width / 2, color="green", ls=":", lw=1)
ax.set_ylabel("% de selecoes"); ax.set_title("Selecao por braco (verde = otimo)"); ax.legend(); plt.show()


C:\Users\Narcelio Filho\AppData\Local\Temp\ipykernel_27908\2850336155.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_ylabel("% de selecoes"); ax.set_title("Selecao por braco (verde = otimo)"); ax.legend(); plt.show()


## 3. Recompensas atrasadas

Comparacao imediato x atrasado (atraso medio = `MEAN_DELAY` rodadas) para TS e UCB.


In [7]:
study = {}
for f in policy_factories(env):
    name = f().name
    if not ("Thompson" in name or "UCB" in name):
        continue
    short = "Thompson" if "Thompson" in name else "UCB1"
    study[(short, "imediato")] = run_policy(f, env, HORIZON, SEEDS, 0.0)
    study[(short, "atrasado")] = run_policy(f, env, HORIZON, SEEDS, MEAN_DELAY)

plt.figure(figsize=(9, 5))
styles = {"imediato": "-", "atrasado": "--"}
for (name, scenario), r in study.items():
    plt.plot(r.mean_cum_regret, styles[scenario], label=f"{name} ({scenario})")
plt.xlabel("rodada"); plt.ylabel("regret acumulado")
plt.title("Impacto do feedback atrasado"); plt.legend(); plt.show()

pd.DataFrame([
    {"politica": k[0], "cenario": k[1], "regret_final": round(r.final_regret_mean, 1),
     "conversao_%": round(r.conversion_rate * 100, 2)}
    for k, r in study.items()
])


C:\Users\Narcelio Filho\AppData\Local\Temp\ipykernel_27908\1215051422.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Impacto do feedback atrasado"); plt.legend(); plt.show()


,politica,cenario,regret_final,conversao_%
0,UCB1,imediato,426.1,15.86
1,UCB1,atrasado,515.7,15.43
2,Thompson,imediato,66.5,17.67
3,Thompson,atrasado,75.7,17.54


## 4. Conclusoes

- **Bandits > regra fixa:** TS e UCB reduzem drasticamente o regret frente ao baseline
  fixo e concentram a selecao no braco otimo — ganho quantitativo da abordagem adaptativa.
- **Thompson Sampling** tende ao melhor desempenho (regret mais baixo, maior % de braco otimo).
- **UCB1 (Nilos-UCB)** equilibra confianca e exploracao via o bonus `sqrt(2 ln t / n_a)`;
  o parametro `c` ajusta o trade-off.
- **Cold-start** tratado por prior bayesiano (TS) e otimismo sob incerteza (UCB).
- **Feedback atrasado** degrada o aprendizado de forma mensuravel, motivando tratamento
  explicito no serving (Etapas 5 e 7).
- **Proximo passo (contexto):** estender para **LinUCB**/TS contextual usando os segmentos
  da Etapa 2 (faixa etaria, historico, regime macro).
